In [1]:
from pathlib import Path
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer

DATA_ROOT = Path.cwd()
DATA_DIR = DATA_ROOT / "data" / "BAN-PL" / "data"
data = pd.read_csv(DATA_DIR / "BAN_PL_1" / "BAN-PL.csv")

data.head()

C:\Users\kubak\Documents\Programming\PycharmProjects\MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,id,Text,Class
0,2200,\n\n\n Polska wtedy oficjalnie powi...,0
1,2201,\n Gigantyczna różnica\n\n{USERNAME}: biorac ...,0
2,2202,\n\n {USERNAME}: Moj kumpel budowla...,0
3,2203,"kura, rzodkiewka za 3pln to nie jest tanio",0
4,2204,{USERNAME}: większość nie idzie w marszu za PO...,0


In [2]:
print(f"Length of dataset: {len(data)}")

class_0 = data['Class'] == 0
class_1 = data['Class'] == 1

print(f"Class 0: {len(data[class_0])}")
print(f"Class 1: {len(data[class_1])}")

Length of dataset: 24000
Class 0: 12000
Class 1: 12000


In [3]:
data_copy = data.copy()

newline = "\n"
nbsp = "\xa0"
username = "{USERNAME}"

data_copy['newline'] = data['Text'].str.count(newline)
data_copy['nbsp'] = data['Text'].str.count(nbsp)
data_copy['username'] = data['Text'].str.count(username)

result = data_copy.groupby('Class')[['newline', 'nbsp', 'username']].mean()

print(result)
print(data_copy.groupby('Class')['username'].value_counts(normalize=True))

        newline      nbsp  username
Class                              
0      1.534083  0.013000     0.821
1      0.975333  0.008333     0.641
Class  username
0      1           0.742917
       0           0.223833
       2           0.025333
       3           0.005250
       4           0.002083
       5           0.000250
       6           0.000250
       7           0.000083
1      1           0.558667
       0           0.410500
       2           0.022750
       3           0.003417
       4           0.001833
       5           0.001083
       6           0.000667
       7           0.000500
       9           0.000167
       10          0.000167
       18          0.000083
       8           0.000083
       12          0.000083
Name: proportion, dtype: float64


In [4]:
data_copy['word_count'] = data_copy['Text'].str.split().str.len()

result = data_copy.groupby('Class')['word_count'].describe()
result

,count,mean,std,min,25%,50%,75%,max
Class,,,,,,,,
0,12000.0,33.782833,43.711942,1.0,11.0,20.0,40.0,791.0
1,12000.0,26.679917,59.135051,1.0,8.0,15.0,29.0,2719.0


In [5]:
duplicate = data_copy['Text'].duplicated().sum()
print(duplicate)

dups = data_copy[data_copy['Text'].duplicated(keep=False)]
labels_per_text = dups.groupby('Text')['Class'].nunique()
print("conflicting:", (labels_per_text > 1).sum())

15
conflicting: 0


In [11]:
tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
ids_to_tokens = tokenizer("k***a")
user = tokenizer("{USERNAME")
print(ids_to_tokens)
print(user)

{'input_ids': [0, 1035, 1881, 1881, 1881, 1011, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
{'input_ids': [0, 1882, 8442, 5382, 6950, 15779, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [19]:
data_copy["n_tokens"] = data_copy["Text"].apply(lambda x: len(tokenizer(x)['input_ids']))
percentiles = data_copy["n_tokens"].quantile([0.5, 0.75, 0.9, 0.95])
for limit in [128, 256]:
    data_copy[f"cut_{limit}"] = data_copy["n_tokens"] > limit
    print(f"max_length={limit}: overall {data_copy[f'cut_{limit}'].mean():.2%}")
    print(data_copy.groupby("Class")[f"cut_{limit}"].mean())

print(percentiles)

max_length=128: overall 6.62%
Class
0    0.078917
1    0.053417
Name: cut_128, dtype: float64
max_length=256: overall 1.60%
Class
0    0.017917
1    0.014083
Name: cut_256, dtype: float64
0.50     34.0
0.75     60.0
0.90    104.0
0.95    148.0
Name: n_tokens, dtype: float64
